In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter
import random
from torchvision import transforms
from PIL import Image

In [2]:
dataset = r"/kaggle/input/chest-xray-pneumonia/chest_xray"

# subderectories
traindata = os.path.join(dataset, "train")
testdata = os.path.join(dataset, "test")
valdata = os.path.join(dataset, "val")


# NORMAL AND PNEUMONIA PATH
Nimage = os.path.join(traindata, "NORMAL")
Pimage = os.path.join(traindata, "PNEUMONIA")


# TOTAl NUMBER OF IMAGES IN TRAINDATA
normal = os.listdir(Nimage)
pneumonia = os.listdir(Pimage)
print(f"TOTAL NUMBER OF IMAGES")
print(f"NORMAL : {len(normal)}")
print(f"PNEUMONIA: {len(pneumonia)}")

TOTAL NUMBER OF IMAGES
NORMAL : 1341
PNEUMONIA: 3875


In [3]:
# Function to check image sizes
def check_image_sizes(folder_path):
    sizes = [(cv2.imread(os.path.join(folder_path, img), cv2.IMREAD_GRAYSCALE).shape[:2])
             for img in os.listdir(folder_path)]
    return sizes, Counter(sizes)


# Check and display sizes for NORMAL and PNEUMONIA
ns, nc = check_image_sizes(Nimage)
ps, pc = check_image_sizes(Pimage)


print("Most common NORMAL dimensions:", nc.most_common(5))
print("Most common PNEUMONIA dimensions:", pc.most_common(5))

Most common NORMAL dimensions: [((1306, 1596), 2), ((1171, 1472), 2), ((994, 1306), 2), ((1356, 1778), 2), ((1279, 1558), 2)]
Most common PNEUMONIA dimensions: [((648, 1072), 7), ((728, 1080), 6), ((704, 1008), 5), ((680, 1008), 5), ((672, 976), 5)]


In [4]:
# Function to resize and pad the images to 224x224
def resize_and_pad(image, target_size=(224, 224)):
    # Get the original image dimensions
    h, w = image.shape[:2]


    # Calculate scaling factor to maintain aspect ratio
    scale = min(target_size[0] / h, target_size[1] / w)
    new_h, new_w = int(h * scale), int(w * scale)
    resized = cv2.resize(image, (new_w, new_h))  # Resize with aspect ratio


    # Calculate padding
    TP = (target_size[0] - new_h) // 2
    BP = target_size[0] - new_h - TP
    LP = (target_size[1] - new_w) // 2
    RP = target_size[1] - new_w - LP


    # Apply padding
    padded = cv2.copyMakeBorder(resized, TP, BP, LP, RP, cv2.BORDER_CONSTANT, value=0)
    return padded


In [5]:
# Function to process images and save them to the upgraded dataset
def process_and_save(input_folder, output_folder, target_size=(224, 224)):
    os.makedirs(output_folder, exist_ok=True)  # Create output folder if it doesn't exist
    for image_name in os.listdir(input_folder):
        image_path = os.path.join(input_folder, image_name)
        output_path = os.path.join(output_folder, image_name)  # Keep filename unchanged
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
        if image is None:
            print(f"Skipping invalid image file: {image_path}")
            continue
        resized_image = resize_and_pad(image, target_size)  # Resize and pad
        cv2.imwrite(output_path, resized_image)  # Save to output folder

In [6]:
# Paths for upgrading dataset
# Udataset = r"D:\SEM 5\Deeplearning\DATASET\UPGRADE_DATASET"
Udataset = "./upgrade_dataset"
subsets = ["train", "test", "val"]


# Process all subsets and categories
for subset in subsets:
    for category in ["NORMAL", "PNEUMONIA"]:
        input_folder = os.path.join(dataset, subset, category)
        output_folder = os.path.join(Udataset, subset, category)
        process_and_save(input_folder, output_folder)

print("Processing completed. Upgraded dataset created.")

Processing completed. Upgraded dataset created.


In [ ]:
# Display a few examples (Optional for verification)
plt.figure(figsize=(6, 5))


# Display original and processed NORMAL image
original_image_path = os.path.join(Nimage, normal[0])
upgraded_image_path = os.path.join(Udataset, "train", "NORMAL", normal[0])

original_image = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)
upgraded_image = cv2.imread(upgraded_image_path, cv2.IMREAD_GRAYSCALE)

plt.subplot(1, 2, 1)
plt.imshow(original_image, cmap='gray')
plt.title("Original NORMAL Image")

plt.subplot(1, 2, 2)
plt.imshow(upgraded_image, cmap='gray')
plt.title("Resized & Padded NORMAL Image")

plt.show()


In [ ]:
# Check and display sizes for NORMAL and PNEUMONIA
normal_sizes, normal_count = check_image_sizes(Nimage)
pneumonia_sizes, pneumonia_count = check_image_sizes(Pimage)

print("Most common NORMAL dimensions:", normal_count.most_common(5))
print("Most common PNEUMONIA dimensions:", pneumonia_count.most_common(5))

In [ ]:
# Scatter plot for both categories
def plot_size_distribution(sizes, label, color):
    heights, widths = zip(*sizes)
    plt.scatter(widths, heights, alpha=0.5, label=label, color=color)


plt.figure(figsize=(8, 6))
plot_size_distribution(normal_sizes, "NORMAL", "blue")
plot_size_distribution(pneumonia_sizes, "PNEUMONIA", "red")
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Dimension Distribution")
plt.legend()
plt.show()

In [ ]:
# Upgraded dataset path
# Udataset = r"D:\SEM 5\Deeplearning\DATASET\UPGRADE_DATASET"


# Subdirectories for upgraded dataset
Utraindata = os.path.join(Udataset, "train")
Utestdata = os.path.join(Udataset, "test")
Uvaldata = os.path.join(Udataset, "val")


# Paths for NORMAL and PNEUMONIA categories in the upgraded dataset
UNtrain = os.path.join(Utraindata, "NORMAL")
UPtrain = os.path.join(Utraindata, "PNEUMONIA")
UNtest = os.path.join(Utestdata, "NORMAL")
UPtest = os.path.join(Utestdata, "PNEUMONIA")
UNval = os.path.join(Uvaldata, "NORMAL")
UPval = os.path.join(Uvaldata, "PNEUMONIA")


# Check and display sizes for NORMAL and PNEUMONIA
normal_sizes, normal_count = check_image_sizes(UNtrain)
pneumonia_sizes, pneumonia_count = check_image_sizes(UPtrain)

print("Most common NORMAL dimensions:", normal_count.most_common(5))
print("Most common PNEUMONIA dimensions:", pneumonia_count.most_common(5))

In [ ]:
# Function to apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
def enhance_contrast(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Read in grayscale
    if img is None:
        print(f"Skipping invalid image: {image_path}")
        return None
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))  # CLAHE settings
    enhanced_img = clahe.apply(img)  # Apply CLAHE enhancement
    return enhanced_img

In [ ]:
# Function to process and save images back to the same folder
def enhance_dataset_images(folder_path):
    for image_name in os.listdir(folder_path):
        image_path = os.path.join(folder_path, image_name)
        enhanced_image = enhance_contrast(image_path)  # Enhance contrast
        if enhanced_image is not None:
            cv2.imwrite(image_path, enhanced_image)  # Overwrite the original image



# Enhance images in all subdirectories
for category in [UNtrain, UPtrain, UNtest, UPtest, UNval, UPval]:
    enhance_dataset_images(category)

In [ ]:
# Display a few examples for verification
def display_images(folder_path, num_images=3):
    images = os.listdir(folder_path)[:num_images]
    plt.figure(figsize=(10, 5))
    for i, img_name in enumerate(images):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        plt.subplot(1, num_images, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(img_name)
        plt.axis('off')


# Example: Display a few enhanced images from train/NORMAL
display_images(UNtrain, num_images=3)
display_images(UPtrain, num_images=3)

In [ ]:
# Function to normalize image pixel values to the range [0, 1]
def normalize_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
    if img is None:
        print(f"Invalid image file: {image_path}")
        return None
    normalized_img = img / 255.0  # Normalize pixel values to [0, 1]
    return (normalized_img * 255).astype(np.uint8)  # Convert back to 8-bit for saving


In [ ]:
# Function to process and normalize images in a folder
def normalize_images(input_folder):
    for image_name in os.listdir(input_folder):
        image_path = os.path.join(input_folder, image_name)
        normalized_image = normalize_image(image_path)
        if normalized_image is not None:
            cv2.imwrite(image_path, normalized_image)  # Save normalized image with the same name

In [ ]:
# Normalize all subdirectories of the upgraded dataset
categories = [UNtrain, UPtrain, UNtest, UPtest, UNval, UPval]

for category in categories:
    normalize_images(category)


    def augment_images(input_folder, output_folder, target_count):
        """
        Augment images in the input folder to match the target count and save them in the output folder.
        """
        os.makedirs(output_folder, exist_ok=True)
        images = os.listdir(input_folder)
        existing_count = len(images)

        if existing_count >= target_count:
            print(f"No augmentation needed. {existing_count} images already present in {input_folder}.")
            return

        # Define augmentation transforms
        augmentation_transforms = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        ])
        # Augment images until the target count is met
        additional_images = target_count - existing_count
        print(f"Augmenting {additional_images} images for {input_folder}...")

        for i in range(additional_images):
            # Randomly pick an image to augment
            img_name = random.choice(images)
            img_path = os.path.join(input_folder, img_name)
            img = Image.open(img_path).convert("RGB")  # Convert to PIL Image

            # Apply augmentation
            augmented_img = augmentation_transforms(img)

            # Save augmented image with a unique name
            new_name = f"{os.path.splitext(img_name)[0]}_aug{i}.png"
            output_path = os.path.join(output_folder, new_name)
            augmented_img.save(output_path)

In [ ]:
 # Paths to dataset folders
UNtrain = os.path.join(Utraindata, "NORMAL")
UPtrain = os.path.join(Utraindata, "PNEUMONIA")

# Define the target number of images for NORMAL and PNEUMONIA
target_count = max(len(os.listdir(UNtrain)), len(os.listdir(UPtrain)))

# Perform augmentation for NORMAL
augment_images(UNtrain, UNtrain, target_count)

In [ ]:
# TOTAl NUMBER OF IMAGES IN TRAINDATA
normal = os.listdir(UNtrain)
pneumonia = os.listdir(UPtrain)
print(f"TOTAL NUMBER OF IMAGES")
print(f"NORMAL : {len(normal)}")
print(f"PNEUMONIA: {len(pneumonia)}")